# EVA Symbolic — Обучение трансформера в многомерном пространстве
## FractalAttention: динамические головы, фрактальные маски, координатные проекции

In [ ]:
# 1. Клонирование + установка
import os
if not os.path.exists('/home/jupyter/EVA'):
    !git clone https://github.com/BlackCatSpb/FCF.git /home/jupyter/EVA
%cd /home/jupyter/EVA
!git pull 2>/dev/null; true
!pip install loguru tokenizers numpy faiss-cpu datasets -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    device = 'cpu'
!df -h /home/jupyter

In [ ]:
# 2. Загрузка символьной модели (учитель = affinity) + датасет
import sys, os, torch, numpy as np
sys.path.insert(0, '/home/jupyter/EVA')
from eva.symbolic import *
from eva.symbolic.fractal_attention import *
from eva.symbolic.advanced_methods import NGramContext
from eva.primordial_layer import PrimordialLayer
from eva.config import FCFConfig

print("Loading affinity matrix (teacher)...")
pf = PotentialField(156, 256)
pf_path = '/home/jupyter/EVA/checkpoints/symbolic/final/potential_field.pt'
if not os.path.exists(pf_path):
    pf_path = '/home/jupyter/EVA/checkpoints/symbolic/step_080000/potential_field.pt'
pf.load_state_dict(torch.load(pf_path, map_location='cpu', weights_only=True))
print(f"Affinity: mean={pf.affinity.mean():.4f}, std={pf.affinity.std():.4f}, max={pf.affinity.max():.4f}")

print("\nCreating transformer (student)...")
config = FCFConfig()
config.d_model = 256; config.vocab_size = 156; config.num_heads = 8; config.max_seq_len = 256
layer = PrimordialLayer(config)
if device == 'cuda': layer = layer.cuda()
print(f"Transformer: {sum(p.numel() for p in layer.parameters()):,} params")

print("\nCreating FractalAttention...")
fractal_attn = FractalAttentionMask(d_model=256, num_base_heads=8, coord_dim_per_level=8, num_levels=4)
if device == 'cuda': fractal_attn = fractal_attn.cuda()

char_vocab = CharacterVocab()
ngram = NGramContext(pf, max_context=4, decay=0.5)

# Dataset
npy_file = '/home/jupyter/EVA/real_data/connected_ru.npy'
if not os.path.exists(npy_file):
    npy_file = '/home/jupyter/EVA/real_data/full_corpus_ids.npy'
all_ids = np.load(npy_file, mmap_mode='r').astype(np.int32)
print(f"Dataset: {len(all_ids)/1e6:.1f}M tokens")

print("\nReady.")

In [ ]:
# 3. Обучение трансформера через knowledge distillation от affinity
import torch.nn.functional as F
import time, gc

BATCH = 128; BLOCK = 128
LR = 1e-4; MAX_STEPS = 50000; LOG_STEP = 2000

optimizer = torch.optim.AdamW(list(layer.parameters()) + list(fractal_attn.parameters()), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_STEPS)

affinity_targets = pf.affinity.to(device)  # [V, V] — soft targets
V = pf.vocab_size
PAD = char_vocab.PAD_IDX

print(f"Training {MAX_STEPS} batches...")
pos = 0; start = time.time(); total_loss = 0

for step in range(1, MAX_STEPS + 1):
    # Form batch
    if pos + BLOCK + 2 > len(all_ids): pos = 0
    ids_batch, lens = [], []
    for _ in range(BATCH):
        if pos + BLOCK + 2 > len(all_ids): pos = 0
        end = min(pos + BLOCK, len(all_ids))
        chunk = all_ids[pos:end]
        sep = np.where((chunk == 0) | (chunk == 3))[0]
        if len(sep) > 0 and sep[0] < BLOCK // 2:
            end = pos + sep[0] + 1; chunk = all_ids[pos:end]
        ids = [int(x) for x in chunk if x >= 0][:BLOCK]
        ids_batch.append(ids); lens.append(len(ids))
        pos += max(len(ids), 32)

    ml = max(lens)
    bt = torch.full((BATCH, ml), PAD, dtype=torch.long, device=device)
    for i, ids in enumerate(ids_batch):
        bt[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)

    # Forward pass через трансформер
    layer.train(); fractal_attn.train()
    x = layer.embed(bt)

    # === LOSS 1: Logit matching (трансформер учится предсказывать как affinity) ===
    hidden = layer.forward_transformer(x)
    logits = layer.forward_logits(hidden)  # [B, L, V]
    
    # Для каждой позиции: target = affinity[текущий_символ, :]
    targets = affinity_targets[bt.clamp(0, V-1)]  # [B, L, V]
    # Mask PAD positions
    mask = (bt != PAD).float().unsqueeze(-1)  # [B, L, 1]
    # KL divergence: softmax(logits) vs affinity targets
    log_probs = F.log_softmax(logits, dim=-1)
    loss_logits = -(targets * log_probs * mask).sum() / (mask.sum() + 1e-8)

    # === LOSS 2: Attention alignment (attention ≈ affinity pattern) ===
    attn = layer.transformer.attention.last_attention  # [B, H, L, L]
    attn_avg = attn.mean(dim=1)[:, :ml, :ml]  # [B, L, L]
    
    # Affinity-based attention target: affinity[bt[i], bt[j]]
    aff_target = affinity_targets[bt[:, :ml, None], bt[:, None, :ml]]  # [B, L, L]
    loss_attn = F.mse_loss(attn_avg, aff_target.detach())

    # === LOSS 3: Fractal attention multi-level ===
    try:
        fractal_out = fractal_attn.multi_level_attention(x[:, :ml, :], bt[:, :ml])
        # Fractal output should match original hidden state (consistency)
        loss_fractal = F.mse_loss(fractal_out, hidden[:, :ml, :].detach())
    except:
        loss_fractal = torch.tensor(0.0, device=device)

    # Composite loss
    loss = loss_logits * 0.4 + loss_attn * 0.3 + loss_fractal * 0.3

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(layer.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(fractal_attn.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    total_loss += loss.item()

    if step % LOG_STEP == 0:
        elapsed = time.time() - start
        avg_l = total_loss / LOG_STEP; total_loss = 0
        bps = step / max(elapsed, 0.01)
        print(f"  step={step} | {bps:.0f} b/s | loss={avg_l:.4f} | "
              f"logits={loss_logits.item():.4f} attn={loss_attn.item():.4f} "
              f"fractal={loss_fractal.item():.4f} | {elapsed/3600:.1f}h")
        gc.collect()
        if device == 'cuda': torch.cuda.empty_cache()

    if step % 10000 == 0:
        torch.save(layer.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/weights.pt')
        torch.save(fractal_attn.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/fractal_attn.pt')
        print(f"  [Saved checkpoint at step {step}]")

torch.save(layer.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/weights_final.pt')
torch.save(fractal_attn.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/fractal_attn_final.pt')
elapsed = time.time() - start
print(f"\nDone: {MAX_STEPS} steps in {elapsed:.0f}s ({elapsed/3600:.1f}h)")
print(f"Speed: {MAX_STEPS/max(elapsed,0.01):.0f} b/s")
!df -h /home/jupyter

In [ ]:
# 4. Тест: генерация с обученным трансформером + FractalAttention
layer.eval(); fractal_attn.eval()

def generate(prompt, max_len=80, temp=0.6):
    ids = char_vocab.encode(prompt)[1:-1]
    ctx = list(ids)
    for _ in range(max_len):
        bt = torch.tensor([ctx[-128:]], dtype=torch.long, device=device)
        x = layer.embed(bt)
        with torch.no_grad():
            hidden = layer.forward_transformer(x)
            logits = layer.forward_logits(hidden)
            # Combine: transformer logits + affinity boost
            transformer_probs = F.softmax(logits[0, -1] / temp, dim=-1)
            affinity_probs = torch.tensor(
                pf.get_continuation_potential(ctx[-1]).cpu().numpy(),
                device=device
            )
            # 70% transformer + 30% affinity
            combined = 0.7 * transformer_probs + 0.3 * F.softmax(affinity_probs / temp, dim=-1)
            next_sym = torch.multinomial(combined, 1).item()
        ctx.append(next_sym)
        if next_sym == char_vocab.EOS_IDX and len(ctx) > len(ids) + 4: break
    return char_vocab.decode(ctx)

print("Generation tests:")
for p in ['pri', 'chelo', 'zem', 'pro', 'kosmo']:
    text = generate(p, max_len=60)
    print(f"  '{p}...' -> '{text[:100]}'")

# Save
!cd /home/jupyter/EVA && tar czf /home/jupyter/transformer_trained.tar.gz checkpoints/transformer/
size = os.path.getsize('/home/jupyter/transformer_trained.tar.gz') / 1024 / 1024
print(f"\nPacked: transformer_trained.tar.gz ({size:.0f} MB)")
print("Download via JupyterLab: right-click -> Download")
!df -h /home/jupyter